# Exercise 03 — Word Embeddings

Words are symbols, but neural networks only understand numbers. In this notebook you turn words into **vectors** that carry meaning, and then put those vectors to work on a real task.

## What you will do
1. **Train your own Word2Vec** model on the Brown corpus and inspect what it learned.
2. **Measure similarity** — implement cosine similarity by hand and check it against gensim.
3. **Explore pre-trained GloVe embeddings** — nearest neighbours, analogies and odd-one-out.
4. **Build a sentiment classifier** on top of GloVe embeddings, train it and evaluate it.
5. **Multiple-choice questions** — to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up — two cells download data.** The Brown corpus (~3 MB) and the GloVe vectors (~130 MB, a minute or two on a decent connection). Start them early so they are ready when you need them.


In [1]:
import numpy as np
from typing import Any

import gensim.downloader
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import tokenize, simple_preprocess

import nltk
from nltk.corpus import brown, stopwords

from tqdm import tqdm
from datasets import load_dataset

import torch
from torch import nn
from torch.utils.data import DataLoader


## 1. Training your own Word2Vec model

To train **Word2Vec embeddings** we first need a text corpus. We use the **Brown Corpus**, a classic collection of American English texts from the 1960s that ships with NLTK.

We also grab NLTK's list of English **stopwords** (*the*, *and*, *is*, …). These words are extremely frequent but carry little meaning, so we filter them out before training and let the model spend its capacity on informative words.


In [2]:
nltk.download("brown")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
print(f"{len(stop_words)} stopwords, e.g. {sorted(stop_words)[:10]}")


198 stopwords, e.g. ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


[nltk_data] Downloading package brown to
[nltk_data]     /Users/mfpetersen/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mfpetersen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Exploring the corpus

Before training anything, look at the data.

**1.1 Fill in the three blanks below to get a feel for the corpus.**

Hints:
- `brown.sents()` returns the corpus as a list of sentences; each sentence is *already* a list of word strings.
- `brown.categories()` returns the genre labels (news, fiction, humor, …).
- To count all word **tokens**, sum the lengths of all the sentences.


In [3]:
sentences = brown.sents()

n_sentences = len(sentences)   # TODO: how many sentences does the corpus have?
n_categories = len(brown.categories())  # TODO: how many categories (genres) does it have?
n_tokens = sum(len(s) for s in sentences)      # TODO: how many word tokens in total?  (hint: sum(len(s) for s in ...))

print(f"Categories : {n_categories}  ->  {brown.categories()}")
print(f"Sentences  : {n_sentences:,}")
print(f"Word tokens: {n_tokens:,}")

print("\nFirst 3 sentences:")
for sent in sentences[:3]:
    print("  ", sent)


Categories : 15  ->  ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']
Sentences  : 57,340
Word tokens: 1,161,192

First 3 sentences:
   ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
   ['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.']
   ['The', 'September-October', 'term', 'jury', 'had', 'been', 'charged', 'by', 'Fulton', 'Superior', 'Court', 'Judge', 'Dur

### Cleaning and preprocessing

Raw sentences need to be **preprocessed** before Word2Vec can use them. For every sentence we:

1. **Join** its words back into a single string — `" ".join(sent)`.
2. **Tokenize, lowercase and deaccent** it with `simple_preprocess(text, deacc=True, min_len=2)`. This also strips punctuation and drops words shorter than 2 characters.
3. **Remove stopwords**, keeping only tokens that are *not* in `stop_words`.

The result is a list of lists: one list of cleaned tokens per sentence — exactly the format `Word2Vec` expects.

**1.2 Build `cleaned_sentences` by applying the three steps above to every sentence in `sentences`.**

Hints:
- A nested list comprehension does this in one expression, but a plain `for` loop is just as good — write whichever you find clearer.
- The corpus has ~57k sentences, so wrap the loop in `tqdm(...)` if you want a progress bar.


In [4]:
cleaned_sentences = []  # TODO: apply steps 1-3 to every sentence in `sentences`
for sent in sentences:
    s = " ".join(sent)
    s = simple_preprocess(s, deacc=True, min_len=2)
    s = list(filter(lambda w: w not in stop_words, s))
    cleaned_sentences.append(s)


print("Raw    :", sentences[0])
print("Cleaned:", cleaned_sentences[0])


Raw    : ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
Cleaned: ['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlanta', 'recent', 'primary', 'election', 'produced', 'evidence', 'irregularities', 'took', 'place']


In [5]:
# ✅ Check your preprocessing
assert isinstance(cleaned_sentences, list), "cleaned_sentences should be a list"
assert len(cleaned_sentences) == len(sentences), "one cleaned sentence per raw sentence"
assert all(isinstance(s, list) for s in cleaned_sentences[:100]), "each element should be a list of tokens"

flat = [w for s in cleaned_sentences for w in s]
assert not any(w in stop_words for w in flat[:5000]), "some stopwords survived the filter"
assert all(w == w.lower() for w in flat[:5000]), "tokens should be lowercased"

print(f"Tokens before cleaning: {sum(len(s) for s in sentences):,}")
print(f"Tokens after cleaning : {len(flat):,}")
print(f"Vocabulary size       : {len(set(flat)):,}")
print("Looks good ✅")


Tokens before cleaning: 1,161,192
Tokens after cleaning : 531,035
Vocabulary size       : 41,096
Looks good ✅


### Training the model

Now train the embeddings. The parameters that matter here:

- `vector_size` — how many dimensions each word vector has.
- `window` — how many words to the left and right count as "context".
- `min_count` — words rarer than this are dropped from the vocabulary.
- `sg` — `1` for **skip-gram** (predict context from the target word), `0` for **CBOW** (predict the target word from its context). Skip-gram works better on small corpora.
- `epochs` — how many passes over the data.

**1.3 Fill in the missing arguments: 50-dimensional vectors, a context window of 3, and the skip-gram objective.**

> Training takes a minute or two.


In [6]:
w2v = Word2Vec(
    sentences=cleaned_sentences,     # TODO: the cleaned corpus
    vector_size=50,   # TODO: 50 dimensions per word
    window=3,        # TODO: 3 words of context on each side
    min_count=1,       # keep all words, even rare ones
    workers=4,         # number of CPU cores to use
    sg=1,            # TODO: skip-gram (1) or CBOW (0)?
    epochs=20,         # passes over the data
    seed=42,
)

print(f"Vocabulary size: {len(w2v.wv):,}")
print(f"Vector size    : {w2v.wv.vector_size}")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary size: 41,096
Vector size    : 50


**1.4 Inspect what the model learned: look up the vector for a word, and find its nearest neighbours.**

Hints:
- `w2v.wv[word]` gives you the vector for a word.
- `w2v.wv.most_similar(word, topn=5)` returns the 5 closest words as `(word, similarity)` pairs.


In [7]:
word = "jury"

vector = w2v.wv[word]      # TODO: the vector for `word`
neighbours = w2v.wv.most_similar(word, topn=5)  # TODO: the 5 most similar words

print(f"Vector for '{word}' (first 10 of {len(vector)} dimensions):")
print(vector[:10])

print(f"\nMost similar to '{word}':")
for w, score in neighbours:
    print(f"   {w:<15} {score:.3f}")


Vector for 'jury' (first 10 of 50 dimensions):
[ 0.7644228  -0.26180187 -0.1405247   0.03392883 -1.0400485   0.51633143
 -0.44002613  0.34843525  0.35911313 -0.07524446]

Most similar to 'jury':
   wexler          0.808
   witnesses       0.794
   karns           0.768
   subpenas        0.756
   huff            0.755


**1.5 Try a few words of your own.** Add two or three words to the list below and look at their neighbours.


In [8]:
for word in ["money", "school", "power", "idiot", "selfish", "christmas"]:  # TODO: add two or three words of your own
    if word in w2v.wv:
        print(f"{word:>12} -> {[w for w, _ in w2v.wv.most_similar(word, topn=5)]}")
    else:
        print(f"{word:>12} -> not in the vocabulary")


       money -> ['colcord', 'hobbies', 'brynge', 'debts', 'borrow']
      school -> ['junior', 'graduate', 'elementary', 'compulsive', 'fortier']
       power -> ['sole', 'holocaust', 'gained', 'arsenal', 'interference']
       idiot -> ['bullshit', 'jollying', 'seekingly', 'bonfiglio', 'debora']
     selfish -> ['rumdum', 'unborn', 'nasty', 'wiser', 'virility']
   christmas -> ['babe', 'evenings', 'stadium', 'lewisohn', 'mornings']


**1.6 Are the neighbours sensible?** The Brown Corpus is about 1 million tokens of 1960s American English. Which kinds of words would you expect this model to handle badly, and why?

---

*Your answer here:*

Words like phone since it wasn't a thing back then or at least not in the same sense as today.

---


### Cosine similarity

`most_similar` ranks words by **cosine similarity** — the cosine of the angle between two vectors:

\begin{align}
\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert \, \lVert \mathbf{v} \rVert}
\end{align}

It ignores the *length* of the vectors and only looks at their **direction**, which is what we want: the value is `1` for vectors pointing the same way, `0` for orthogonal ones, and `-1` for opposite ones.

**1.7 Implement `cosine_similarity` yourself.**

Hints:
- `np.dot(u, v)` for the dot product in the numerator.
- `np.linalg.norm(u)` for the length $\lVert \mathbf{u} \rVert$.


In [9]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    # TODO: implement the formula above
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))


In [10]:
# ✅ Check your implementation against gensim's own
for w1, w2 in [("jury", "court"), ("man", "woman"), ("jury", "car")]:
    mine = cosine_similarity(w2v.wv[w1], w2v.wv[w2])
    theirs = w2v.wv.similarity(w1, w2)
    assert np.isclose(mine, theirs, atol=1e-5), f"mismatch for ({w1}, {w2}): {mine} vs {theirs}"
    print(f"cos({w1:>5}, {w2:<6}) = {mine:+.4f}   (gensim: {theirs:+.4f})")

print("Looks good ✅")


cos( jury, court ) = +0.5899   (gensim: +0.5899)
cos(  man, woman ) = +0.7621   (gensim: +0.7621)
cos( jury, car   ) = +0.2094   (gensim: +0.2094)
Looks good ✅


**1.8 Which of the three pairs above is the most similar, and which the least? Does that match your intuition?**

---

*Your answer here:*

man and woman are most similar, i would maybe have expected jury and court to be more similar,
but it makes good sense that jury and car is the least similar pair.

---


## 2. Pre-trained GloVe embeddings

Training on 1 million tokens gets you only so far. In practice we usually start from **pre-trained embeddings** learned on far more text.

Here we load **GloVe** (Global Vectors for Word Representation), trained on Wikipedia + Gigaword — about 6 billion tokens. Each word is mapped to a **100-dimensional** vector.

> The download is ~130 MB and is cached, so it is only slow the first time.


In [11]:
glove_vectors: KeyedVectors = gensim.downloader.load("glove-wiki-gigaword-100")

print(f"Vocabulary size: {len(glove_vectors):,}")
print(f"Vector size    : {glove_vectors.vector_size}")


Vocabulary size: 400,000
Vector size    : 100


**2.1 Compare your own embeddings with GloVe's.** Print the 5 nearest neighbours of the same word according to each model.

Hint: `glove_vectors` has the same interface as `w2v.wv` — `most_similar`, `similarity`, and `glove_vectors[word]` all work.


In [12]:
word = "jury"

print(f"Your Word2Vec (Brown, ~1M tokens):")
print("  ", w2v.wv.most_similar(word, topn=5))  # TODO: the 5 nearest neighbours according to `w2v.wv`

print(f"GloVe (Wikipedia + Gigaword, 6B tokens):")
print("  ", glove_vectors.most_similar(word, topn=5))  # TODO: the 5 nearest neighbours according to `glove_vectors`


Your Word2Vec (Brown, ~1M tokens):
   [('wexler', 0.8083228468894958), ('witnesses', 0.7939268946647644), ('karns', 0.7679564952850342), ('subpenas', 0.7563773989677429), ('huff', 0.75544673204422)]
GloVe (Wikipedia + Gigaword, 6B tokens):
   [('jurors', 0.8421859741210938), ('judge', 0.7821869850158691), ('trial', 0.7602095007896423), ('court', 0.7458882331848145), ('verdict', 0.7383376955986023)]


**2.2 The two lists are different. Name two reasons why.** Think about *how much* text each model saw, and *what kind* of text it was.

---

*Your answer here:*

So glove have more data and therefore more context for each word, which makes it more likely to find similar words.

---


### Analogies

The famous property of word embeddings is that **directions in the vector space carry meaning**. The vector that takes you from *man* to *king* is roughly the same one that takes you from *woman* to *queen*:

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$

`most_similar` can do this arithmetic for you: words in `positive=[...]` are added, words in `negative=[...]` are subtracted.

**2.3 Implement `analogy(a, b, c)` — read as "*a* is to *b* as *c* is to ...?" — which computes $b - a + c$.**

Hint: that is `most_similar(positive=[b, c], negative=[a], topn=topn)`.


In [13]:
def analogy(a: str, b: str, c: str, topn: int = 3) -> list:
    """a is to b as c is to ...?   ->   b - a + c"""
    # TODO: use glove_vectors.most_similar with the right positive/negative words
    return glove_vectors.most_similar(positive=[b, c], negative=[a], topn=topn)


print("man   -> king   ::  woman ->", [w for w, _ in analogy("man", "king", "woman")])
print("paris -> france ::  tokyo ->", [w for w, _ in analogy("paris", "france", "tokyo")])
print("good  -> better ::  bad   ->", [w for w, _ in analogy("good", "better", "bad")])
print("money -> rich   ::  poor  ->", [w for w, _ in analogy("money", "rich", "poor")])
# TODO: invent one analogy of your own and test it here


man   -> king   ::  woman -> ['queen', 'monarch', 'throne']
paris -> france ::  tokyo -> ['japan', 'korea', 'japanese']
good  -> better ::  bad   -> ['worse', 'too', 'even']
money -> rich   ::  poor  -> ['impoverished', 'poorer', 'especially']


### Odd one out

Since we can measure similarity, we can also spot the word that does not belong: `doesnt_match` returns the word furthest from the average of the group.

**2.4 Complete the loop and add a group of your own.**


In [14]:
groups = [
    ["breakfast", "lunch", "dinner", "football"],
    ["denmark", "sweden", "norway", "guitar"],
    ["red", "green", "blue", "monday"],
    ["Windows", "Unix", "Linux", "penguin"]
]

for group in groups:
    odd = glove_vectors.doesnt_match(group)  # TODO: which word does not belong?  (hint: glove_vectors.doesnt_match)
    print(f"{str(group):<50} -> {odd}")


['breakfast', 'lunch', 'dinner', 'football']       -> football
['denmark', 'sweden', 'norway', 'guitar']          -> guitar
['red', 'green', 'blue', 'monday']                 -> monday
['Windows', 'Unix', 'Linux', 'penguin']            -> penguin


### A word of caution

Embeddings learn whatever regularities are present in their training text — including social stereotypes. Run the cell below and look at what comes out.


In [15]:
print("man -> doctor     ::  woman ->", [w for w, _ in analogy("man", "doctor", "woman", topn=5)])
print("man -> programmer ::  woman ->", [w for w, _ in analogy("man", "programmer", "woman", topn=5)])
print("man -> boss       ::  woman ->", [w for w, _ in analogy("man", "boss", "woman", topn=5)])


man -> doctor     ::  woman -> ['nurse', 'physician', 'doctors', 'patient', 'dentist']
man -> programmer ::  woman -> ['educator', 'programmers', 'linguist', 'technician', 'freelance']
man -> boss       ::  woman -> ['bosses', 'girlfriend', 'boyfriend', 'colleague', 'lover']


**2.5 What do you see, and where does it come from?** Why is this a problem if these embeddings are used as features in, say, a CV-screening system?

*(We come back to this in the Responsible AI session.)*

---

*Your answer here:*

---


## 3. Using GloVe embeddings to train a classifier

So far we have looked at individual words. Now we use embeddings as **features** for a downstream task: classifying the sentiment of tweets.

We use the **Sentiment140** dataset, where each tweet is labelled `1` (positive) or `0` (negative). The full dataset has 1.6M tweets; we take a subset to keep things fast.


In [16]:
ds = load_dataset("adilbekovich/Sentiment140Twitter")

train = ds["train"].select(range(50_000))
test = ds["test"].select(range(10_000))

print(f"Training set size: {len(train):,}")
print(f"Test set size    : {len(test):,}")

print("\nA few examples:")
for row in train.select(range(3)):
    print(f"  label={row['label']}  {row['text'][:90]}")


Training set size: 50,000
Test set size    : 10,000

A few examples:
  label=0  Closet organizer install complete.  Now for the torture part- going through the mounds of 
  label=1  Mornin' All!!  ....I need to wake up....this will get me goin'!!   â« http://blip.fm/~7qg
  label=0  @Lega_c ahhhhhhhhhh! he suxxxxxxx! he claimed it! 


### From a sentence to a vector

A neural network needs a **fixed-length** vector per example, but tweets have different lengths. The simplest fix is to **average the word vectors** of every word in the sentence.

It is a crude representation — it throws away word order completely — but it is a surprisingly strong baseline.

**3.1 Implement `sentence_embedding`.**

Steps:
1. **Tokenize** the sentence: `list(tokenize(sentence, deacc=True, to_lower=True))`.
2. **Look up** the vector of every token the model knows. Use `w in model` to test membership — tweets are full of typos and hashtags that GloVe has never seen.
3. **Average** the vectors with `np.mean(..., axis=0)`. If *no* word was known, return a **zero vector** of length `model.vector_size` instead (averaging an empty list would fail).


In [17]:
def sentence_embedding(sentence: str, model: Any) -> np.ndarray:
    """Average the embeddings of all known words in `sentence`."""
    tokens = list(tokenize(sentence, deacc=True, to_lower=True))        # TODO: step 1
    word_vectors = [model.get_vector(w) for w in tokens if w in model] # TODO: step 2
    # TODO: step 3 — the mean of `word_vectors`, or a zero vector if it is empty
    return np.mean(word_vectors, axis=0) if word_vectors else np.zeros(model.vector_size, dtype=model.vectors.dtype)


In [18]:
# ✅ Check your sentence_embedding
v = sentence_embedding("i love this movie", glove_vectors)
assert v.shape == (100,), f"expected shape (100,), got {v.shape}"
expected = np.mean([glove_vectors[w] for w in ["i", "love", "this", "movie"]], axis=0)
assert np.allclose(v, expected, atol=1e-5), "should be the *average* of the word vectors"

unknown = sentence_embedding("zzzzqqq wwwwxyzzz", glove_vectors)
assert unknown.shape == (100,) and np.allclose(unknown, 0), "unknown-only sentences should give a zero vector"

# Sentences with a similar meaning should end up closer together
a = sentence_embedding("the food was delicious", glove_vectors)
b = sentence_embedding("the meal tasted great", glove_vectors)
c = sentence_embedding("my laptop battery died", glove_vectors)
print(f"similar pair  : {cosine_similarity(a, b):.3f}")
print(f"unrelated pair: {cosine_similarity(a, c):.3f}")
assert cosine_similarity(a, b) > cosine_similarity(a, c)

print("Looks good ✅")


similar pair  : 0.887
unrelated pair: 0.597
Looks good ✅


**3.2 Add an `"embeddings"` column to both splits by applying `sentence_embedding` to the `"text"` column.**

Hints:
- `dataset.map(fn)` applies `fn` to every row; `fn` receives the row as a dict and returns a dict of **new** columns.
- So: `train.map(lambda x: {"embeddings": sentence_embedding(x["text"], glove_vectors)})`.
- `set_format(type="torch", ...)` is already written for you — it makes the dataset hand back PyTorch tensors.

> This takes a minute or so for 60k tweets.


In [19]:
def to_embedding(x):
    return {"embeddings": sentence_embedding(x["text"], glove_vectors)}
train = train.map(to_embedding)  # TODO: map the training set to add an "embeddings" column
test = test.map(to_embedding)   # TODO: same for the test set

train.set_format(type="torch", columns=["embeddings", "label"])
test.set_format(type="torch", columns=["embeddings", "label"])

print(train)


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'embeddings'],
    num_rows: 50000
})


### The classifier

A small feed-forward network — the same building blocks you implemented by hand last week:

1. **Input**: the 100-dimensional averaged embedding of a tweet.
2. **Hidden layer**: `Linear(input_dim, 256)` followed by a **ReLU**.
3. **Output layer**: `Linear(256, 1)` followed by a **Sigmoid**, so the output is a probability in $(0, 1)$ — the probability that the tweet is positive.

**3.3 Fill in the layers and the forward pass, then create the model with the right input dimension.**

Hint: the input dimension is the size of a GloVe vector — `glove_vectors.vector_size`.


In [20]:
class SentimentClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_features=input_dim, out_features=hidden_dim)      # TODO: Linear, input_dim -> hidden_dim
        self.relu = torch.nn.ReLU()     # TODO: ReLU activation
        self.fc2 = torch.nn.Linear(in_features=hidden_dim, out_features=1)      # TODO: Linear, hidden_dim -> 1
        self.sigmoid = torch.nn.Sigmoid()  # TODO: Sigmoid activation

    def forward(self, x):
        # TODO: fc1 -> relu -> fc2 -> sigmoid
        MLP = torch.nn.Sequential(
            self.fc1, self.relu, self.fc2, self.sigmoid
        )
        return MLP(x)


classifier = SentimentClassifier(glove_vectors.vector_size)  # TODO: create the model with the correct input dimension


In [21]:
# ✅ Check your classifier
dummy = torch.randn(8, glove_vectors.vector_size)
out = classifier(dummy)

assert out.numel() == 8, f"expected one output per example, got shape {tuple(out.shape)}"
assert (out >= 0).all() and (out <= 1).all(), "outputs should be probabilities — did you apply the sigmoid?"

print(classifier)
print(f"\nTrainable parameters: {sum(p.numel() for p in classifier.parameters()):,}")
print("Looks good ✅")


SentimentClassifier(
  (fc1): Linear(in_features=100, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Trainable parameters: 26,113
Looks good ✅


### Training setup

- **Batch size 64** — how many tweets we process before each parameter update.
- **30 epochs** — how many times we go through the whole training set.
- **`BCELoss`** — binary cross-entropy, the standard loss when the model outputs a single probability.
- **SGD, lr = 0.01** — the optimizer that applies the updates.


In [22]:
BATCH_SIZE = 64
EPOCHS = 30

loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(classifier.parameters(), lr=0.01)

train_dataloader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test, batch_size=BATCH_SIZE)


**3.4 Write a small `accuracy` helper.** The model outputs probabilities, but accuracy needs hard 0/1 decisions.

Hints:
- Threshold at 0.5: `(outputs > 0.5)` gives a boolean tensor — `.float()` turns it into 0s and 1s.
- Then compare with `labels` and take the **mean** of the matches.


In [23]:
def accuracy(outputs: torch.Tensor, labels: torch.Tensor) -> float:
    """Fraction of correct predictions. `outputs` are probabilities in (0, 1)."""
    predictions = outputs > 0.5  # TODO: threshold the probabilities at 0.5
    return torch.mean((predictions == labels).float()).item()         # TODO: the fraction of predictions that match `labels`


In [24]:
# ✅ Check your accuracy function
probs = torch.tensor([0.9, 0.2, 0.6, 0.4])
labels = torch.tensor([1.0, 0.0, 0.0, 0.0])
assert np.isclose(float(accuracy(probs, labels)), 0.75), f"expected 0.75, got {accuracy(probs, labels)}"
print("Looks good ✅")


Looks good ✅


### Training the classifier

The loop is the same one you saw last week. For each batch:

1. **Forward pass** — run the embeddings through the model.
2. **Compute the loss** — how far the predictions are from the labels.
3. **Backward pass** — `zero_grad()`, `backward()`, `step()`.

**3.5 Fill in the three steps.**

Hint: the model returns a tensor of shape `(batch_size, 1)` while the labels have shape `(batch_size,)`. Use `.squeeze()` on the outputs so `BCELoss` gets matching shapes.

> Training 30 epochs over 50k tweets takes around 10 minutes on a CPU. Start it and read ahead.


In [25]:
classifier.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for batch in train_dataloader:
        inputs = batch["embeddings"].float()
        labels = batch["label"].float()

        # TODO: 1. forward pass
        outputs = classifier(inputs)

        # TODO: 2. compute the loss (remember to squeeze the outputs)
        loss = loss_fn(outputs.squeeze(), labels)

        # TODO: 3. zero the gradients, backpropagate, and update the weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1:>2}/{EPOCHS}, Loss: {epoch_loss / len(train_dataloader):.4f}")


Epoch  1/30, Loss: 0.6854
Epoch  2/30, Loss: 0.6704
Epoch  3/30, Loss: 0.6519
Epoch  4/30, Loss: 0.6325
Epoch  5/30, Loss: 0.6160
Epoch  6/30, Loss: 0.6036
Epoch  7/30, Loss: 0.5951
Epoch  8/30, Loss: 0.5893
Epoch  9/30, Loss: 0.5849
Epoch 10/30, Loss: 0.5815
Epoch 11/30, Loss: 0.5792
Epoch 12/30, Loss: 0.5770
Epoch 13/30, Loss: 0.5750
Epoch 14/30, Loss: 0.5734
Epoch 15/30, Loss: 0.5721
Epoch 16/30, Loss: 0.5709
Epoch 17/30, Loss: 0.5705
Epoch 18/30, Loss: 0.5694
Epoch 19/30, Loss: 0.5685
Epoch 20/30, Loss: 0.5682
Epoch 21/30, Loss: 0.5674
Epoch 22/30, Loss: 0.5666
Epoch 23/30, Loss: 0.5662
Epoch 24/30, Loss: 0.5657
Epoch 25/30, Loss: 0.5651
Epoch 26/30, Loss: 0.5646
Epoch 27/30, Loss: 0.5642
Epoch 28/30, Loss: 0.5638
Epoch 29/30, Loss: 0.5634
Epoch 30/30, Loss: 0.5629


### Evaluation

**3.6 Evaluate the trained model on the test set.**

Remember to:
- put the model in **evaluation mode**,
- run the loop inside **`torch.no_grad()`** — no gradients are needed here,
- accumulate the loss and the accuracy per batch, then divide by the number of **batches**.


In [27]:
classifier.eval()
total_loss, total_acc = 0.0, 0.0

with torch.no_grad():
    for batch in test_dataloader:
        inputs = batch["embeddings"].float()
        labels = batch["label"].float()

        outputs = classifier(inputs).squeeze()       # TODO: forward pass
        total_loss += loss_fn(outputs, labels).item()   # TODO: the loss for this batch, as a plain number
        total_acc += accuracy(outputs, labels)    # TODO: the accuracy for this batch

print(f"Test loss    : {total_loss / len(test_dataloader):.4f}")
print(f"Test accuracy: {100 * total_acc / len(test_dataloader):.2f}%")


Test loss    : 0.5635
Test accuracy: 71.39%


### Try it on your own sentences

**3.7 Write `predict_sentiment(text)`, which returns the probability that `text` is positive.**

Hints:
- Embed the text with `sentence_embedding(text, glove_vectors)`.
- Turn it into a tensor: `torch.tensor(vec).float()`.
- The model expects a **batch**, so add a leading dimension with `.unsqueeze(0)`.
- Wrap the call in `torch.no_grad()` and use `.item()` to get a plain float back.


In [36]:
def predict_sentiment(text: str) -> float:
    """Probability that `text` expresses positive sentiment."""
    # TODO: embed -> tensor -> add batch dimension -> run through the classifier
    embedded = sentence_embedding(text, glove_vectors)
    tensor = torch.tensor(embedded).float()
    with torch.no_grad():
        return classifier(tensor.unsqueeze(0)).item()


for text in [
    "i love this, best day ever",
    "worst service i have ever had, never coming back",
    "the weather is nice today",
    # TODO: add a couple of sentences of your own
    "I feel like Garfield about this Monday",
    "I'm looking forward to Weekend.",
]:
    print(f"{predict_sentiment(text):.2f}   {text}")


0.91   i love this, best day ever
0.11   worst service i have ever had, never coming back
0.64   the weather is nice today
0.46   I feel like Garfield about this Monday
0.86   I'm looking forward to Weekend.
0.54   ?


### Where this representation breaks down

Averaging word vectors throws away **word order**. The two sentences below contain exactly the same words.


In [37]:
a = "the movie was good, not bad at all"
b = "the movie was bad, not good at all"

print(f"{predict_sentiment(a):.3f}   {a}")
print(f"{predict_sentiment(b):.3f}   {b}")


0.335   the movie was good, not bad at all
0.335   the movie was bad, not good at all


**3.8 Explain what you see.** Why does the model give these two sentences (almost) the same score, and what would a model need in order to tell them apart?

---

*Your answer here:*

The reason is that the model most likely weigh the word bard quite high which then makes the sentiment negative.

It needs word ordering, as a minimum to preserve "not bad" / "not good" to be able to know that the meaning of these
should be inverted.
This could be achieved with a context aware embedding like BERT.

---


## 4. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 4.1. Purpose of Word Embeddings

What is the main purpose of word embeddings in NLP?

A. To convert words into high-dimensional one-hot vectors<br>
B. To map words into continuous vector spaces that capture semantic meaning<br>
C. To remove stopwords from text before processing<br>
D. To reduce the training time of convolutional networks<br>

**Answer:**
B.

---

### 4.2. One-Hot vs. Embeddings

Compared to one-hot encoding, word embeddings:

A. Have the same dimensionality as the vocabulary size<br>
B. Provide dense, low-dimensional representations that capture similarities<br>
C. Are always manually designed by experts<br>
D. Cannot be trained with neural networks<br>

**Answer:**
B.

---

### 4.3. Word2Vec Models

The Skip-gram model in Word2Vec is designed to:

A. Predict the context words given a target word<br>
B. Predict the target word given the context words<br>
C. Cluster words into fixed categories<br>
D. Remove rare words from the corpus<br>

**Answer:**
A.

---

### 4.4. Embedding Matrix Shape

In a neural network with vocabulary size $V$ and embedding dimension $d$, the embedding matrix has shape:

A. $(d \times V)$<br>
B. $(V \times d)$<br>
C. $(V \times V)$<br>
D. $(d \times d)$<br>

**Answer:**
B.

---

### 4.5. Semantic Relationships

Word embeddings can capture analogies such as:

A. king – man + woman ≈ queen<br>
B. dog – cat + car ≈ airplane<br>
C. apple – red + fast ≈ running<br>
D. chair – table + sky ≈ cloud<br>

**Answer:**
A.

---

### 4.6. Contextual vs. Static Embeddings

How do contextual embeddings (e.g., BERT) differ from static embeddings (e.g., Word2Vec)?

A. They assign the same vector to a word regardless of context<br>
B. They assign different vectors to a word depending on its context<br>
C. They are always lower-dimensional than static embeddings<br>
D. They do not require pretraining on large corpora<br>

**Answer:**
B.

---

### 4.7. Sparse vs. Dense Representations

Compared to Bag-of-Words (BoW) vectors, neural word embeddings are:

A. Higher dimensional and sparse<br>
B. Always binary representations<br>
C. Lower dimensional and sparse<br>
D. Lower dimensional and dense<br>

**Answer:**
D.

---

### 4.8. Cosine Similarity

Why is cosine similarity, rather than Euclidean distance, the usual way to compare two word vectors?

A. It is the only similarity measure that can be computed efficiently<br>
B. It compares the direction of the vectors and ignores their magnitude<br>
C. It always returns a value between 0 and 1<br>
D. It requires the vectors to be normalised in advance<br>

**Answer:**
B.

---

### 4.9. CBOW vs. Skip-gram

The Continuous Bag of Words (CBOW) model aims to:

A. Predict the target word given its surrounding context words<br>
B. Assign unique one-hot vectors to words<br>
C. Predict the context words given a target word<br>
D. Cluster words into topics using SVD<br>

**Answer:**
A.

---

### 4.10. Distributional Semantics

The idea that "you shall know a word by the company it keeps" refers to:

A. Overfitting in NLP models<br>
B. Context-based learning of embeddings<br>
C. Sentence segmentation<br>
D. Stopword removal<br>

**Answer:**
B.
